In [ ]:
# Llama-3.2 is gated. Store a Kaggle secret named HF_TOKEN.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=token)
    print('Hugging Face login: OK')
except Exception as exc:
    print('HF login was not completed automatically:', exc)
    print('If the model is already cached and accessible, you can continue.')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:
# TODO for Question 4
"""

Hint:
# Analyze the weights of the query projection layer (q_proj) where layer = k
 q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

# Apply SVD to decompose the weight matrix into U (left singular vectors), S (singular values), and V (right singular vectors).
 U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

# Compute the retained energy ratio (Cumulative energy) of each singular value.
energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
"""

import torch
import matplotlib.pyplot as plt

LAYERS = [0, 7, 15]
TOP_K = 200
results = {}

for k in LAYERS:
    q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()
    U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

    energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100

    # 累積能量第一次達到 50% 的位置；索引從 0 開始，所以 rank 要加 1
    min_rank = int((energy_ratios >= 50).nonzero()[0].item()) + 1

    results[k] = {'S': S, 'energy': energy_ratios, 'min_rank': min_rank}
    print(f'Layer {k}: minimum target rank for >= 50% energy = {min_rank}')


In [ ]:
# 每一層各畫一張長條圖和一張折線圖，排成 3 x 2
fig, axes = plt.subplots(len(LAYERS), 2, figsize=(12, 10))

for row, k in enumerate(LAYERS):
    S = results[k]['S'][:TOP_K].cpu().numpy()
    energy = results[k]['energy'][:TOP_K].cpu().numpy()
    idx = range(1, len(S) + 1)

    ax_bar = axes[row, 0]
    ax_bar.bar(idx, S)
    ax_bar.set_title(f'Layer {k}: Singular Value Magnitude')
    ax_bar.set_xlabel('Singular Value Index')
    ax_bar.set_ylabel('Magnitude')

    ax_line = axes[row, 1]
    ax_line.plot(idx, energy)
    ax_line.axhline(50, linestyle='--', color='gray')
    ax_line.set_title(f'Layer {k}: Cumulative Energy')
    ax_line.set_xlabel('Target Rank')
    ax_line.set_ylabel('Cumulative Energy (%)')

plt.tight_layout()
plt.savefig('q4_svd_analysis.png', dpi=200)
plt.show()